# Réseau des éditions : graveurs, plaques et copies

Les trois premiers notebooks répondent chacun à une seule question, en restant volontairement
simples : circulation géographique (`01`), nuage par ville (`02`), filiation des plaques
(`03`). Une collègue a vu ces visualisations et en veut une qui **rassemble** ces relations
en un seul graphe — quitte à ce que ce soit dense, tant que ça reste explorable.

Ce notebook construit donc un **graphe en réseau** (nœuds = éditions, liens = relations entre
elles) sur l'ensemble du corpus (19 villes), en combinant deux relations déjà calculées dans
les autres notebooks, mais jamais affichées ensemble :
- le **réemploi de plaques** d'une édition à l'autre (même logique que `03_reemploi_plaques.ipynb`) ;
- les **copies** d'une édition envers le travail d'un autre graveur (même logique que
  `01_carte_circulation.ipynb`).

Contrairement aux trois autres notebooks (positions calculées à l'avance en Python), la
disposition des nœuds est ici calculée **dans le navigateur**, par une simulation de forces
([D3.js](https://d3js.org/)) : les nœuds reliés s'attirent, les autres se repoussent, jusqu'à
un équilibre. C'est la première fois que ce projet utilise D3 — les trois premiers notebooks
n'en avaient pas besoin (positions géographiques ou chronologiques déjà connues), mais une
disposition de réseau n'a pas de position "naturelle" a priori.

Même source de données que les trois autres notebooks :
`retours_celine/BNU_corpus.ods` (feuille `Synthèse`).

In [1]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "reseau_editions.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS que les trois autres notebooks. Contrairement à `03_reemploi_plaques.ipynb`
(qui gardait le nom de graveur brut), on réutilise ici `normaliser_graveur` de
`01_carte_circulation.ipynb` (qui retire les dates entre parenthèses et la virgule finale) :
les liens de copie de `01` sont résolus par rapprochement de texte contre un registre de noms
normalisés, donc les deux relations (plaques et copies) doivent partager la **même** identité
de graveur pour pouvoir être combinées dans un seul graphe. Chaque édition reçoit aussi un
identifiant unique (`id`), utilisé comme nœud du graphe.

In [2]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que les
    trois autres notebooks)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def categorie_technique(t):
    t = (t or "").strip().lower()
    if t == "bois":
        return "bois"
    if t == "cuivre":
        return "cuivre"
    return "inconnue"

def normaliser_graveur(nom):
    """Retire les dates entre parenthèses et la ponctuation superflue (identique à
    01_carte_circulation.ipynb)."""
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.strip().rstrip(",").strip()
    return re.sub(r"\s+", " ", nom)

def graveur_identifiable(nom):
    """Un 'AnonymeXXXX' est un identifiant valable ; seules les mentions non identifiantes
    ('?', 'inaccessible', case vide) sont écartées (identique à 01_carte_circulation.ipynb)."""
    if not nom:
        return False
    if nom.strip().lower() in {"?", "inaccessible"}:
        return False
    return True

# Ville canonique -> (variantes rencontrées dans le tableau), identique à
# 01_carte_circulation.ipynb — sert uniquement à colorer les nœuds par ville ici (pas de
# carte dans ce notebook), donc les éditions non localisables gardent leur ville brute
# plutôt que d'être écartées : ça ne les empêche pas de participer au réseau.
CORRESPONDANCE_VILLES = {
    "Amsterdam": "Amsterdam", "Anvers": "Anvers", "Anvers / Rotterdam": "Anvers",
    "Arnhemii": "Arnhem", "Augsbourg": "Augsbourg", "Bruxelles": "Bruxelles",
    "Cöllen": "Cologne", "[Köln]": "Cologne",
    "Francfort": "Francfort", "Francfort-sur-le-Main": "Francfort",
    "Leyde": "Leyde", "Londres": "Londres", "Lyon": "Lyon",
    "Mayence (Meinz)": "Mayence", "Nuremberg": "Nuremberg",
    "Paris": "Paris", "[Paris]": "Paris", "Rouen": "Rouen",
    "Tusculanum": "Toscolano", "Valladolid": "Valladolid", "Venise": "Venise",
    "Vienne": "Vienne", "Vienne\xa0?": "Vienne", "[Haarlem]": "Haarlem",
}

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

editions = []
for row in corpus:
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    ville_brute = row.get("ville", "").strip()
    graveur_brut = row.get(COL_GRAVEUR, "").strip()
    editions.append({
        "id": len(editions),
        "ville": CORRESPONDANCE_VILLES.get(ville_brute, ville_brute or "Ville inconnue"),
        "annee": annee,
        "titre": titre.strip(),
        "technique": categorie_technique(row.get("technique", "")),
        "graveur": normaliser_graveur(graveur_brut) if graveur_identifiable(graveur_brut) else None,
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
        "copies": row.get("copies de cette édition", "").strip(),
    })

print(len(editions), "éditions au total")
print(len({e['graveur'] for e in editions if e['graveur']}), "jeux de plaques (graveurs) identifiables")
print(len({e['ville'] for e in editions}), "villes distinctes")

111 éditions au total
49 jeux de plaques (graveurs) identifiables
21 villes distinctes


## 2. Liens "réemploi de plaques"

Même logique que `03_reemploi_plaques.ipynb` : pour chaque graveur identifiable apparaissant
dans au moins 2 éditions, on relie ses éditions par ordre chronologique, et chaque lien est
classé réimpression / transmission / incertain selon que l'éditeur change ou non (et selon
qu'il est lui-même identifié). Seule différence avec `03` : les liens pointent ici vers des
**éditions** précises (`id`), pas seulement vers une paire (année, éditeur), puisque ces
éditions deviennent des nœuds du graphe.

In [3]:
groupes_plaques = {}
for e in editions:
    if e["graveur"] is None:
        continue
    groupes_plaques.setdefault(e["graveur"], []).append(e)

def editeur_fiable(pub):
    """Un éditeur non identifié ('s.n.', case vide) ne permet pas de dire avec certitude si
    deux éditions se succèdent chez le même éditeur ou changent de main."""
    return pub not in {"s.n.", "Éditeur non identifié"}

def type_lien_plaque(pub1, pub2):
    if not (editeur_fiable(pub1) and editeur_fiable(pub2)):
        return "incertain"
    return "reprise" if pub1 == pub2 else "transfert"

liens_plaques = []
for graveur, eds in groupes_plaques.items():
    eds_tries = sorted(eds, key=lambda e: e["annee"])
    for a, b in zip(eds_tries, eds_tries[1:]):
        liens_plaques.append({
            "source": a["id"], "target": b["id"],
            "type": type_lien_plaque(a["publisher"], b["publisher"]),
            "graveur": graveur,
        })

from collections import Counter
compte = Counter(l["type"] for l in liens_plaques)
print(len(liens_plaques), "liens de réemploi de plaques :", dict(compte))

61 liens de réemploi de plaques : {'transfert': 35, 'reprise': 17, 'incertain': 9}


## 3. Liens "copie"

Même logique que `01_carte_circulation.ipynb` (colonne `copies de cette édition`) : mentions
ambiguës ("ou", "?") écartées sans deviner, candidats extraits et rapprochés d'un registre de
graveurs connus par chevauchement de mots, puis liés à l'édition antérieure la plus récente de
ce graveur. Seule différence avec `01` : le registre est construit sur **toutes les éditions**
du corpus (pas seulement celles localisables sur les 19 villes canoniques), et chaque lien
pointe vers une édition précise plutôt qu'une paire (ville, année).

In [4]:
def normaliser_candidat_anonyme(c):
    """« Anonyme 1563 » ou un simple « 1572 » -> 'AnonymeXXXX' (même convention que le registre)."""
    c = re.sub(r"anonyme\s*(\d{4})", r"Anonyme\1", c, flags=re.IGNORECASE)
    if re.fullmatch(r"\d{4}", c.strip()):
        c = "Anonyme" + c.strip()
    return c.strip()

def eclater_enumeration(fragment):
    """« Anonyme1563, 1572 » -> ['Anonyme1563', '1572'] si tout ressemble à une liste d'années."""
    parties = [p.strip() for p in fragment.split(",")]
    if len(parties) > 1 and all(re.fullmatch(r"(anonyme\s*)?\d{4}", p, re.IGNORECASE) for p in parties):
        return parties
    return [fragment]

def mention_ambigue(texte):
    """"ou" ou "?" signale une attribution hésitante entre plusieurs graveurs : on ne devine pas."""
    return bool(re.search(r"\bou\b", texte, re.IGNORECASE)) or "?" in texte

def extraire_candidats_copie(texte):
    """« copie X et Y » / « même famille que X, Y et Z » -> liste de noms de graveurs candidats."""
    t = re.sub(r"\(.*?\)", "", texte)
    t = re.sub(r"^\s*(copie|même famille que)\s*", "", t, flags=re.IGNORECASE)
    bruts = re.split(r"\s+et\s+", t)
    candidats = []
    for c in bruts:
        c = c.strip(" .,;?\xa0")
        if not c:
            continue
        for sous in eclater_enumeration(c):
            sous = normaliser_candidat_anonyme(sous.strip(" .,;?\xa0"))
            if sous:
                candidats.append(sous)
    return candidats

def tokens_nom(nom):
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.replace(",", " ")
    return set(t.lower() for t in re.findall(r"[a-zà-öø-ÿ']+", nom) if len(t) >= 3)

def trouver_graveur_connu(candidat, registre):
    """Fait correspondre un nom en texte libre à une clé du registre (chevauchement de mots)."""
    if candidat in registre:
        return candidat
    tc = tokens_nom(candidat)
    if not tc:
        return None
    meilleur, meilleur_score = None, 0
    for nom_reg in registre:
        if nom_reg.lower().startswith("anonyme"):
            continue  # déjà couvert par la correspondance exacte ci-dessus
        score = len(tc & tokens_nom(nom_reg))
        if score > meilleur_score:
            meilleur, meilleur_score = nom_reg, score
    return meilleur if meilleur_score >= 1 else None

# registre : graveur normalisé -> liste de TOUTES ses éditions (pas seulement la 1ère)
registre_graveur = {}
for e in editions:
    if e["graveur"] is None:
        continue
    registre_graveur.setdefault(e["graveur"], []).append(e)

liens_copies = []
non_resolus = []
for e in editions:
    if not e["copies"]:
        continue

    if mention_ambigue(e["copies"]):
        non_resolus.append((e, e["copies"], "(toute la mention)", "attribution ambiguë (ou/possibilité multiple)"))
        continue

    candidats = extraire_candidats_copie(e["copies"])
    matches_uniques = {}
    for c in candidats:
        match = trouver_graveur_connu(c, registre_graveur)
        if match is None:
            non_resolus.append((e, e["copies"], c, "aucune correspondance"))
        else:
            matches_uniques.setdefault(match, []).append(c)

    for match in matches_uniques:
        instances_anterieures = [
            inst for inst in registre_graveur[match]
            if inst["annee"] <= e["annee"] and inst["id"] != e["id"]
        ]
        if not instances_anterieures:
            non_resolus.append((e, e["copies"], match, "pas d'édition antérieure de ce graveur"))
            continue
        origine = max(instances_anterieures, key=lambda inst: inst["annee"])
        liens_copies.append({"source": origine["id"], "target": e["id"], "type": "copie", "graveur": match})

print(len(liens_copies), "liens de copie résolus")
print(len(non_resolus), "mentions non résolues :")
for e, texte, cible, raison in non_resolus:
    print(f"  ✗ {e['ville']} {e['annee']} « {e['titre'][:50]} » — {texte!r} — {cible!r} ({raison})")

24 liens de copie résolus
10 mentions non résolues :
  ✗ Toscolano 1526 « P. Ovidii Metamorphosis » — 'copie QUOI\xa0?' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1527 « Publii Ovidii Nasonis Sulmonensis Metamorphoseos L » — 'copie Leroy II, Guillaume ou 1497' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1532 « Le Grand Olympe » — 'copie Leroy II, Guillaume ou Anonyme1497 ou Anonyme1527' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1556 « Trois Premiers livres de la Métamorphose d'Ovide » — 'copie Bernard Salomon' — 'Salomon, Bernard' (pas d'édition antérieure de ce graveur)
  ✗ Cologne 1602 « Metamorphoseon Ovidianarum per Crispianum Passaeum » — 'copie van der Borcht\xa0? Salomon\xa0?' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Paris 1669 « Les Métamorphoses d´Ovide traduites par T. Corneil » — 'copie Clein, Francisco (inv.) et Savery, Salomon (sc

## 4. Assemblage du graphe

On ne garde que les éditions ayant **au moins un lien** (plaque ou copie) : une édition isolée
n'a rien à montrer dans un graphe de relations, elle ajouterait du bruit sans information (un
point de plus, nulle part relié). Chaque nœud reçoit une couleur propre à sa ville, par
rotation à l'angle d'or (même principe que `02_nuage_editions_villes.ipynb` pour les
éditeurs/graveurs) — pas de palette manuelle à maintenir pour 21 villes. La taille du nœud
suit son degré (nombre de liens), pour repérer d'un coup d'œil les éditions les plus centrales
dans le réseau.

In [5]:
import colorsys

def couleur_categorielle(i, saturation=0.55, luminosite=0.48):
    """Couleur hex distincte pour l'index i, par rotation à l'angle d'or (voir
    02_nuage_editions_villes.ipynb pour l'explication du principe)."""
    teinte = (i * 137.508 % 360) / 360
    r, g, b = colorsys.hls_to_rgb(teinte, luminosite, saturation)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))

liens = liens_plaques + liens_copies

degre = Counter()
for l in liens:
    degre[l["source"]] += 1
    degre[l["target"]] += 1

editions_par_id = {e["id"]: e for e in editions}
ids_connectes = sorted(degre.keys())

villes_du_reseau = sorted({editions_par_id[i]["ville"] for i in ids_connectes})
couleur_ville = {v: couleur_categorielle(i) for i, v in enumerate(villes_du_reseau)}

noeuds = []
for i in ids_connectes:
    e = editions_par_id[i]
    noeuds.append({
        "id": i,
        "ville": e["ville"],
        "annee": e["annee"],
        "titre": e["titre"],
        "technique": e["technique"],
        "editeur": e["publisher"],
        "graveur": e["graveur"],
        "lien": e["lien"],
        "degre": degre[i],
        "couleur": couleur_ville[e["ville"]],
        "rayon": round(5 + min(degre[i], 12) ** 0.5 * 2.5, 1),
    })

print(len(noeuds), "nœuds (sur", len(editions), "éditions au total) —", len(liens), "liens")
print(len(villes_du_reseau), "villes représentées dans le réseau")
plus_connectee = max(noeuds, key=lambda n: n["degre"])
print("Nœud le plus connecté :", plus_connectee["titre"][:50], f"({plus_connectee['ville']}, {plus_connectee['annee']}) — degré {plus_connectee['degre']}")

95 nœuds (sur 111 éditions au total) — 85 liens
18 villes représentées dans le réseau
Nœud le plus connecté : La vita et metamorfoseo d'Ovidio (Lyon, 1584) — degré 6


## 5. Génération du graphe (HTML autonome, D3.js)

[D3.js](https://d3js.org/) v7, via CDN comme Leaflet/MapLibre dans les autres notebooks —
`d3-force` pour la disposition, `d3.drag` pour déplacer un nœud à la souris, `d3.zoom` pour
naviguer. Contrairement à un graphe en réseau "libre" (position purement issue de la
simulation, sans signification), l'**abscisse est ici l'année** de l'édition : chaque nœud est
tiré horizontalement vers sa position chronologique par une force `forceX` dédiée, avec un axe
gradué (1500-1700) en bas du graphe. L'**ordonnée reste libre** — uniquement pilotée par la
répulsion et la collision, pour étaler verticalement les nœuds d'une même période et éviter
qu'ils ne se chevauchent — sans signification propre. Résultat : un graphe qui garde la
lecture "qui est relié à qui" d'un réseau, mais où l'on peut aussi lire "quand", ce que la
disposition libre d'origine ne permettait pas. Légende des 4 types de lien, infobulle
identique aux trois autres notebooks (survol = aperçu sans lien, clic = épinglée avec le lien
"voir"), et un tableau dépliable listant toutes les relations pour l'accessibilité.

In [6]:
LIBELLES_TECHNIQUE = {"bois": "Bois", "cuivre": "Cuivre", "inconnue": "Technique inconnue"}
LIBELLES_TYPE = {"reprise": "Réimpression (même éditeur)", "transfert": "Transmission à un autre éditeur",
                  "incertain": "Incertain (éditeur non identifié)", "copie": "Copie du travail d'un autre graveur"}

def texte_edition(e):
    return f'{e["titre"]} — {e["ville"]}, {e["annee"]} ({e["publisher"]})'

def lignes_tableau():
    lignes = []
    for l in sorted(liens, key=lambda l: (l["type"], editions_par_id[l["source"]]["annee"])):
        source, cible = editions_par_id[l["source"]], editions_par_id[l["target"]]
        lignes.append(
            f'<tr><td>{LIBELLES_TYPE[l["type"]]}</td><td>{l["graveur"]}</td>'
            f'<td>{texte_edition(source)}</td><td>{texte_edition(cible)}</td></tr>'
        )
    return "\n".join(lignes)

LIGNES_TABLEAU = lignes_tableau()

TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Réseau des éditions : graveurs, plaques et copies</title>
<script src="https://unpkg.com/d3@7/dist/d3.min.js"></script>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6; --c-transfert: #c0392b; --c-copie: #16a085;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5; --c-transfert: #e0685a; --c-copie: #2fcf9f;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1080px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 12px; }

  .legende { display:flex; gap:16px; flex-wrap:wrap; font-size:12px; margin:0 0 6px; }
  .legende .item { display:flex; align-items:center; gap:6px; }
  .legende .trait { display:inline-block; width:26px; height:0; border-top-width:2px; }
  .legende .reprise { border-top:2px solid var(--texte-att); }
  .legende .transfert { border-top:2px solid var(--c-transfert); }
  .legende .incertain { border-top:2px dashed var(--texte-att); }
  .legende .copie { border-top:2px dashed var(--c-copie); }
  .note-graphe { font-size:12px; color:var(--texte-att); font-style:italic; margin:0 0 12px; }

  .cadre-graphe { border:1px solid var(--trait); border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.15);
    overflow:hidden; touch-action:none; }
  #graphe { display:block; width:100%; }
  .lien { fill:none; }
  .lien-reprise { stroke:var(--texte-att); stroke-width:1.4; }
  .lien-transfert { stroke:var(--c-transfert); stroke-width:1.8; }
  .lien-incertain { stroke:var(--texte-att); stroke-width:1.4; stroke-dasharray:3,3; }
  .lien-copie { stroke:var(--c-copie); stroke-width:1.8; stroke-dasharray:6,3; }
  .noeud { stroke:var(--contour-point); stroke-width:1.2; cursor:pointer; fill-opacity:.9; }
  .noeud:hover, .noeud.actif { fill-opacity:1; stroke-width:2; }
  .grille-annee { stroke:var(--trait); stroke-width:1; }
  .ligne-base-annee { stroke:var(--texte-att); stroke-width:1.5; }
  .etiquette-annee { font-size:10px; fill:var(--texte-att); }

  .action-tableau { margin:14px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { width:100%; border-collapse:collapse; font-size:12px; margin:8px 0;
    display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }

  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
  .infobulle.epinglee { pointer-events:auto; }
  .infobulle a { color:var(--c-lien); }
  .infobulle .fermer-infobulle { position:absolute; top:2px; right:6px; cursor:pointer;
    color:var(--texte-att); font-size:13px; }
</style></head><body>
<div class="page">
  <h1>Réseau des éditions : graveurs, plaques et copies</h1>
  <p class="souschapo">Un nœud = une édition, colorée par ville. Un lien = une relation entre
    deux éditions (réemploi des mêmes plaques, ou copie du travail d'un autre graveur).
    Cliquer un nœud affiche le détail avec le lien "voir" (le survol seul n'affiche qu'un
    aperçu).</p>
  <div class="legende">
    <div class="item"><span class="trait reprise"></span>réimpression (même éditeur)</div>
    <div class="item"><span class="trait transfert"></span>transmission à un autre éditeur</div>
    <div class="item"><span class="trait incertain"></span>incertain (éditeur non identifié)</div>
    <div class="item"><span class="trait copie"></span>copie du travail d'un autre graveur</div>
  </div>
  <p class="note-graphe">Abscisse = année (voir l'axe en bas du graphe) · ordonnée libre,
    pour étaler les nœuds d'une même période et éviter les chevauchements · couleur = ville
    (voir infobulle) · taille = nombre de liens · glisser un nœud pour le déplacer, molette
    pour zoomer.</p>
  <div class="cadre-graphe"><div id="graphe"></div></div>
  <div class="action-tableau">
    <button class="bascule" id="boutonTableau">Afficher le tableau détaillé</button>
    <table class="tableau-detaille" id="tableauDetaille">
      <thead><tr><th>Type</th><th>Graveur / plaques</th><th>Édition source</th><th>Édition cible</th></tr></thead>
      <tbody>
        __LIGNES_TABLEAU__
      </tbody>
    </table>
  </div>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const noeuds = __NOEUDS__;
  const liens = (__LIENS__).map(l => ({...l}));
  const libellesTechnique = __LIBELLES_TECHNIQUE__;

  function contenuApercu(p) {
    return '<b>' + p.titre + '</b><br>' +
      '<span>' + p.ville + ', ' + p.annee + ' · ' + libellesTechnique[p.technique] + '</span>' +
      '<br><i>' + p.editeur + '</i>' +
      (p.graveur ? '<br>Plaques : ' + p.graveur : '');
  }
  function contenuDetaille(p) {
    const lien = p.lien ? '<br><a href="' + p.lien + '" target="_blank">→ voir</a>' : '';
    return contenuApercu(p) + lien;
  }

  const cadre = document.getElementById('graphe');
  const largeur = cadre.clientWidth || 960;
  const hauteur = 700;

  const svg = d3.select('#graphe').append('svg')
    .attr('viewBox', [0, 0, largeur, hauteur])
    .attr('width', '100%').attr('height', hauteur);

  const zoomCouche = svg.append('g');
  svg.call(d3.zoom().scaleExtent([0.3, 4]).on('zoom', (ev) => {
    zoomCouche.attr('transform', ev.transform);
  }));

  // --- Axe des années : l'abscisse d'un nœud est pilotée par une force qui le tire vers
  // xAnnee(d.annee) (voir plus bas), donc l'axe doit utiliser exactement la même échelle. ---
  const anneeMin = 1490, anneeMax = 1750;
  const margeAxe = {gauche: 30, droite: 30, haut: 10, bas: 28};
  function xAnnee(annee) {
    const t = (annee - anneeMin) / (anneeMax - anneeMin);
    return margeAxe.gauche + t * (largeur - margeAxe.gauche - margeAxe.droite);
  }

  // Flèches (marqueurs), une par type de lien orienté — même principe que les flèches de
  // 01_carte_circulation.ipynb et 03_reemploi_plaques.ipynb, en SVG pur.
  const defs = svg.append('defs');
  function ajouterMarqueur(id, couleurVar) {
    defs.append('marker')
      .attr('id', id).attr('viewBox', '0 0 10 10').attr('refX', 8).attr('refY', 5)
      .attr('markerWidth', 6).attr('markerHeight', 6).attr('orient', 'auto-start-reverse')
      .append('path').attr('d', 'M0,0 L10,5 L0,10 z').attr('fill', couleurVar);
  }
  ajouterMarqueur('fleche-transfert', 'var(--c-transfert)');
  ajouterMarqueur('fleche-copie', 'var(--c-copie)');

  // L'axe fait partie de zoomCouche (comme les liens et les nœuds) : il zoome/se déplace
  // avec le contenu, pour que les positions restent alignées sur l'échelle à tout niveau de zoom.
  const coucheAxe = zoomCouche.append('g');
  const y_base_axe = hauteur - margeAxe.bas;
  coucheAxe.append('line')
    .attr('x1', margeAxe.gauche).attr('y1', y_base_axe)
    .attr('x2', largeur - margeAxe.droite).attr('y2', y_base_axe)
    .attr('class', 'ligne-base-annee');
  [1500, 1600, 1700].forEach(a => {
    const x = xAnnee(a);
    coucheAxe.append('line')
      .attr('x1', x).attr('y1', margeAxe.haut).attr('x2', x).attr('y2', y_base_axe)
      .attr('class', 'grille-annee');
    coucheAxe.append('text')
      .attr('x', x).attr('y', y_base_axe + 16)
      .attr('text-anchor', 'middle').attr('class', 'etiquette-annee').text(a);
  });

  const coucheLiens = zoomCouche.append('g');
  const coucheNoeuds = zoomCouche.append('g');

  const traitLien = coucheLiens.selectAll('line').data(liens).join('line')
    .attr('class', d => 'lien lien-' + d.type)
    .attr('marker-end', d => (d.type === 'transfert' || d.type === 'copie') ? `url(#fleche-${d.type})` : null);

  // L'abscisse de chaque nœud est tirée fermement vers sa position chronologique (forceX,
  // forte intensité) ; l'ordonnée reste libre, seulement contrainte par la répulsion et la
  // collision (pour étaler verticalement les nœuds d'une même période sans se chevaucher) et
  // une force de rappel très faible vers le centre (pour ne pas dériver indéfiniment).
  const simulation = d3.forceSimulation(noeuds)
    .force('lien', d3.forceLink(liens).id(d => d.id).distance(35).strength(0.2))
    .force('charge', d3.forceManyBody().strength(-55))
    .force('x', d3.forceX(d => xAnnee(d.annee)).strength(0.9))
    .force('y', d3.forceY(hauteur / 2).strength(0.06))
    .force('collision', d3.forceCollide().radius(d => d.rayon + 3));

  const cercleNoeud = coucheNoeuds.selectAll('circle').data(noeuds).join('circle')
    .attr('class', 'noeud')
    .attr('r', d => d.rayon)
    .attr('fill', d => d.couleur)
    .call(d3.drag()
      .on('start', (ev, d) => { if (!ev.active) simulation.alphaTarget(0.3).restart(); d.fx = d.x; d.fy = d.y; })
      .on('drag', (ev, d) => { d.fx = ev.x; d.fy = ev.y; })
      .on('end', (ev, d) => { if (!ev.active) simulation.alphaTarget(0); d.fx = null; d.fy = null; }));

  // Sans cette contrainte, un nœud fortement repoussé peut sortir du cadre visible (rogné
  // par overflow:hidden sur .cadre-graphe) et devenir invisible et impossible à cliquer —
  // repéré en testant le graphe avec un clic automatisé qui manquait sa cible.
  simulation.on('tick', () => {
    noeuds.forEach(d => {
      d.x = Math.max(d.rayon, Math.min(largeur - d.rayon, d.x));
      d.y = Math.max(d.rayon, Math.min(hauteur - margeAxe.bas - d.rayon, d.y));
    });
    traitLien
      .attr('x1', d => d.source.x).attr('y1', d => d.source.y)
      .attr('x2', d => d.target.x).attr('y2', d => d.target.y);
    cercleNoeud.attr('cx', d => d.x).attr('cy', d => d.y);
  });

  // --- Infobulle : identique aux trois autres notebooks (survol = aperçu qui suit la
  // souris, sans lien ; clic = épinglée sur place, avec le lien "voir", pointer-events actifs). ---
  const infobulle = document.getElementById('infobulle');
  const page = document.querySelector('.page');
  let infobulleEpinglee = false;

  function positionnerInfobulle(ev) {
    const r = page.getBoundingClientRect();
    infobulle.style.left = (ev.clientX - r.left + 14) + 'px';
    infobulle.style.top = (ev.clientY - r.top + 14) + 'px';
  }
  function fermerInfobulle() {
    infobulleEpinglee = false;
    infobulle.classList.remove('epinglee');
    infobulle.style.opacity = 0;
    cercleNoeud.classed('actif', false);
  }

  cercleNoeud
    .on('mouseenter', function (ev, d) {
      if (infobulleEpinglee) return;
      d3.select(this).classed('actif', true);
      infobulle.innerHTML = contenuApercu(d);
      infobulle.style.opacity = 1;
    })
    .on('mousemove', function (ev) {
      if (infobulleEpinglee) return;
      positionnerInfobulle(ev);
    })
    .on('mouseleave', function () {
      if (infobulleEpinglee) return;
      d3.select(this).classed('actif', false);
      infobulle.style.opacity = 0;
    })
    .on('click', function (ev, d) {
      ev.stopPropagation();
      positionnerInfobulle(ev);
      infobulle.innerHTML = contenuDetaille(d) + '<span class="fermer-infobulle" title="Fermer">×</span>';
      infobulle.style.opacity = 1;
      infobulle.classList.add('epinglee');
      infobulleEpinglee = true;
      cercleNoeud.classed('actif', false);
      d3.select(this).classed('actif', true);
      infobulle.querySelector('.fermer-infobulle').addEventListener('click', fermerInfobulle);
    });

  document.addEventListener('click', (ev) => {
    if (infobulleEpinglee && !infobulle.contains(ev.target) && !ev.target.classList.contains('noeud')) {
      fermerInfobulle();
    }
  });

  const boutonTableau = document.getElementById('boutonTableau');
  boutonTableau.addEventListener('click', () => {
    const tableau = document.getElementById('tableauDetaille');
    const visible = tableau.classList.toggle('visible');
    boutonTableau.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__LIGNES_TABLEAU__", LIGNES_TABLEAU)
    .replace("__NOEUDS__", json.dumps(noeuds, ensure_ascii=False))
    .replace("__LIENS__", json.dumps(liens, ensure_ascii=False))
    .replace("__LIBELLES_TECHNIQUE__", json.dumps(LIBELLES_TECHNIQUE, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Réseau écrit dans", CHEMIN_SORTIE)

Réseau écrit dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/reseau_editions.html
